# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore, process, and analyze the FAIR<sup>2</sup> mass spectrometry dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset is defined with a Croissant schema, available at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```



In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {metadata.version}\nLicense: {metadata.license}")
print(f"Published: {metadata.datePublished}")


## 2. Data Overview

Let's review available record sets, fields, and their Croissant `@id`s.

We will list the record sets, explore one as an example, and print the field/column names and their `@id`s to inform subsequent steps.

In [ ]:
# List record sets and their IDs
record_sets = metadata.record_sets
print(f"This dataset has {len(record_sets)} record set(s).")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    # List fields for this record set
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}) | Data type: {field.data_type}")
    print("")
# We'll select the first record set for detailed examination
if len(record_sets):
    main_recordset_id = record_sets[0].id
    print(f"Using RecordSet '@id': {main_recordset_id}\n")
else:
    main_recordset_id = None

## 3. Data Extraction

Now we'll load data from each record set into a DataFrame and view sample field names and records. **All access is via each entity's `@id`.**

In [ ]:
# Extract data from each record set
dataframes = {}

for rs in record_sets:
    print(f"Loading records for RecordSet: {rs.name} (@id={rs.id})")
    # Using @id as the identifier for record_set parameter
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Loaded {len(df)} records. Columns (fields by @id):")
    print(f"  {list(df.columns)}\n")
    # Show head
    display(df.head(2))

# Use main_recordset_id for further examples
if main_recordset_id is not None:
    main_df = dataframes[main_recordset_id]
    print(f"Sample fields in main record set: {list(main_df.columns)}\n")
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)

We'll:
- Select a relevant numeric field (by `@id`)
- Filter records where this numeric field is above a threshold
- Normalize the field
- Optionally group by a categorical field, if present

Let's inspect the available numeric fields in our main record set.

In [ ]:
# Identify candidate numeric fields by @id
numeric_fields = []
chosen_numeric_field_id = None

for field in record_sets[0].fields:
    if field.data_type in ('schema:Number', 'schema:Float', 'schema:Integer', 'Number', 'Float', 'Integer'):
        numeric_fields.append((field.name, field.id, field.data_type))
print("Numeric fields available:")
for name, fid, dtype in numeric_fields:
    print(f"  - {name} (@id={fid}) -- {dtype}")

# Pick the first numeric field for demonstration
if numeric_fields:
    chosen_numeric_field_id = numeric_fields[0][1]
    print(f"\nSelected for analysis: {chosen_numeric_field_id}\n")
else:
    print("No numeric field found.")

In [ ]:
# EDA: filter, normalize, group
if main_recordset_id is not None and chosen_numeric_field_id is not None:
    df = dataframes[main_recordset_id]
    # Handle possible missing/non-numeric values
    df[chosen_numeric_field_id] = pd.to_numeric(df[chosen_numeric_field_id], errors='coerce')
    threshold = df[chosen_numeric_field_id].mean() if not df[chosen_numeric_field_id].isnull().all() else 0

    filtered_df = df[df[chosen_numeric_field_id] > threshold].copy()
    print(f"Filtered records with {chosen_numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df.head())

    # Normalization (z-score)
    mean = filtered_df[chosen_numeric_field_id].mean()
    std = filtered_df[chosen_numeric_field_id].std()
    filtered_df[f"{chosen_numeric_field_id}_normalized"] = (filtered_df[chosen_numeric_field_id] - mean) / std
    print(f"Normalized {chosen_numeric_field_id} for filtered records:")
    display(filtered_df[[chosen_numeric_field_id, f"{chosen_numeric_field_id}_normalized"]].head())

    # Try to group by a categorical field (not the numeric one)
    # Pick first non-numeric field
    group_field_id = None
    for field in record_sets[0].fields:
        if field.id != chosen_numeric_field_id and field.data_type not in (
                'schema:Number', 'schema:Float', 'schema:Integer', 'Number', 'Float', 'Integer'):
            group_field_id = field.id
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[chosen_numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {chosen_numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found for grouping.")
else:
    print("No numeric field available for analysis.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and, if available, its normalized form and a group comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_recordset_id is not None and chosen_numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[chosen_numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {chosen_numeric_field_id}")
    plt.xlabel(chosen_numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Show normalized if created
    if f"{chosen_numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[f"{chosen_numeric_field_id}_normalized"].dropna(), kde=True)
        plt.title(f"Normalized {chosen_numeric_field_id} (Z-score)")
        plt.xlabel(f"{chosen_numeric_field_id}_normalized")
        plt.ylabel('Count')
        plt.show()

    # Bar chart of group means (if present)
    if 'grouped_df' in locals() and group_field_id in grouped_df.columns:
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df[chosen_numeric_field_id])
        plt.title(f"Mean {chosen_numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {chosen_numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load the Croissant dataset and metadata using `mlcroissant`
- Explore record sets and field structures via their `@id`
- Load and examine tabular data into pandas DataFrames
- Filter, normalize, and aggregate numerical data using its Croissant semantic `@id`
- Visualize summary statistics and distributions

**Key takeaways:**
- Croissant `@id` references ensure robust and reproducible analysis pipelines across data and schema updates.
- The FAIR<sup>2</sup> mass spectrometry dataset exposes protein-level quantification and annotation for advanced downstream analysis.
- For advanced analytics, you can repeat the workflow for different record sets or fields as needed.

Explore further and adapt EDA or ML code to your use case!